# 24-тарау. Қазақша жаңалықтарды тақырыпқа бөлеміз

*«Машина қалай үйренеді?» кітабына арналған блокнот. Әр ұяшықтың түсіндірмесі кітапта берілген.*

Ұяшықтарды жоғарыдан төмен қарай ретімен іске қосыңыз: **Shift + Enter**.

## 24.1. Жобаның мақсаты

## 24.2. 1-қадам. Деректерді жүктеу

In [ ]:
import pandas as pd

URL = ("https://raw.githubusercontent.com/amirtemir1996n3-dotcom/"
       "mashina-qalai-uirenedi/main/data/news_kz.csv")
df = pd.read_csv(URL)
print(df.shape)
print(df["санат"].value_counts())

In [ ]:
for title, cat in df.sample(5, random_state=2).values:
    print(f"{cat:>10} | {title}")

## 24.3. 2-қадам. Оқу және тест жиындары

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["тақырып"], df["санат"], test_size=0.25,
    stratify=df["санат"], random_state=42)
print(len(X_train), len(X_test))
print(y_test.value_counts().tolist())

## 24.4. 3-қадам. Сөздер ме, әріп тіркестері ме?

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

word_model = make_pipeline(
    TfidfVectorizer(),
    LogisticRegression(C=10, max_iter=1000))
word_model.fit(X_train, y_train)
print(f"Сөздер: {word_model.score(X_test, y_test):.1%}")

In [ ]:
model = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5)),
    LogisticRegression(C=10, max_iter=1000))
model.fit(X_train, y_train)
print(f"Әріп тіркестері: {model.score(X_test, y_test):.1%}")

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

classifiers = {
    "Аңғал Байес": MultinomialNB(alpha=0.1),
    "Логистикалық регрессия": LogisticRegression(C=10, max_iter=1000),
    "Сызықтық SVM": LinearSVC(),
}
for name, clf in classifiers.items():
    pipe = make_pipeline(
        TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5)), clf)
    scores = cross_val_score(pipe, df["тақырып"], df["санат"], cv=5)
    print(f"{name:23} {scores.mean():.1%} ± {scores.std():.1%}")

## 24.5. 4-қадам. Деректер көбейсе ше? Оқу қисығы

In [ ]:
import numpy as np
from sklearn.model_selection import learning_curve

sizes, train_sc, val_sc = learning_curve(
    model, df["тақырып"], df["санат"], cv=5,
    train_sizes=np.linspace(0.1, 1.0, 7))
for n, s in zip(sizes, val_sc.mean(axis=1)):
    print(f"{n:4d} мысал → {s:.1%}")

## 24.6. 5-қадам. Модель қай жерде қателеседі?

In [ ]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, digits=2))

In [ ]:
res = pd.DataFrame({"тақырып": X_test, "шын": y_test, "модель": y_pred})
errors = res[res["шын"] != res["модель"]]
print("Қате жіктелгені:", len(errors))
for _, r in errors.head(6).iterrows():
    print(r["тақырып"])
    print(f"    шын санаты: {r['шын']}, модель: {r['модель']}")

## 24.7. 6-қадам. Модель нені үйренді?

In [ ]:
vec = model.named_steps["tfidfvectorizer"]
clf = model.named_steps["logisticregression"]
names = vec.get_feature_names_out()
for i, cat in enumerate(clf.classes_):
    order = np.argsort(clf.coef_[i])[::-1]      # салмақ кему ретімен
    grams = [names[j] for j in order if len(names[j].strip()) == 5]
    print(f"{cat:>10}: {', '.join(grams[:6])}")

## 24.8. 7-қадам. Жаңа тақырыптарды жіктеу

In [ ]:
new = [
    "Жас боксшылар әлем чемпионатынан үш медаль әкелді",
    "Ұлттық банк теңгені қолдау үшін валюта сатты",
    "Оқушылар жасанды интеллект көмегімен робот құрастырды",
    "Астанада ұлттық аспаптар фестивалі өтеді",
    "Дәрігерлер күзде тұмауға қарсы егу керек деп ескертті",
    "Футболшыларға арналған жаңа медициналық орталық ашылды",
]
proba = model.predict_proba(new)
for title, p in zip(new, proba):
    best = np.argsort(p)[::-1][:2]
    c1, c2 = clf.classes_[best]
    print(f"{title}\n    → {c1} {p[best[0]]:.0%}, {c2} {p[best[1]]:.0%}")